In [0]:
"""
06_factory_dashboard.py

Streaming Factory Dashboard.

Creates a single executive manufacturing dashboard by combining
the Gold KPI tables.

Inputs:
    machine_kpis
    quality_kpis
    production_kpis
    material_kpis
    packaging_kpis

Output:
    factory_dashboard

Author:
Sumanth Vempalle

Version:
2.1.0
"""

import dlt

from pyspark.sql.functions import (
    avg,
    current_timestamp,
    sum,
)


# ============================================================
# Factory Dashboard
# ============================================================

@dlt.table(
    name="factory_dashboard",
    comment="Executive manufacturing dashboard summarizing factory KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dlt.expect(
    "operations_available",
    "operations_completed > 0",
)

@dlt.expect(
    "tests_available",
    "tests_completed > 0",
)

def factory_dashboard():

    machine = dlt.read(
        "machine_kpis"
    )

    quality = dlt.read(
        "quality_kpis"
    )

    production = dlt.read(
        "production_kpis_batch"
    )

    material = dlt.read(
        "material_kpis"
    )

    packaging = dlt.read(
        "packaging_kpis"
    )

    machine_summary = (

        machine

        .agg(

            sum(
                "operations_completed"
            ).alias(
                "operations_completed"
            ),

            avg(
                "pass_rate"
            ).alias(
                "machine_pass_rate"
            ),

            avg(
                "average_cycle_time_sec"
            ).alias(
                "average_cycle_time_sec"
            ),

            avg(
                "average_force_kn"
            ).alias(
                "average_force_kn"
            ),

        )

    )

    quality_summary = (

        quality

        .agg(

            sum(
                "tests_completed"
            ).alias(
                "tests_completed"
            ),

            avg(
                "pass_rate"
            ).alias(
                "quality_pass_rate"
            ),

        )

    )

    production_summary = (

        production

        .agg(

            sum(
                "work_orders_created"
            ).alias(
                "work_orders_created"
            ),

            sum(
                "executions_started"
            ).alias(
                "executions_started"
            ),

            sum(
                "products_started"
            ).alias(
                "products_started"
            ),

        )

    )

    material_summary = (

        material

        .agg(

            sum(
                "materials_scanned"
            ).alias(
                "materials_scanned"
            ),

            avg(
                "scan_success_rate"
            ).alias(
                "material_scan_success_rate"
            ),

        )

    )

    packaging_summary = (

        packaging

        .agg(

            sum(
                "packages_completed"
            ).alias(
                "packages_completed"
            ),

            avg(
                "shipment_readiness_rate"
            ).alias(
                "shipment_readiness_rate"
            ),

        )

    )

    return (

        machine_summary

        .crossJoin(
            quality_summary
        )

        .crossJoin(
            production_summary
        )

        .crossJoin(
            material_summary
        )

        .crossJoin(
            packaging_summary
        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )